# CMS Medicare Inpatient Data Profiling

This notebook profiles the validated 2024 raw snapshot directly from Azurite. Source strings remain unchanged in `raw_df`; numeric conversions are made only in a separate analytical copy.

## Load Validated Raw Snapshot

In [ ]:
import hashlib
import json
import os
from pathlib import Path

import pandas as pd
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(project_root / '.env')
year = os.environ.get('CMS_REPORTING_YEAR', '2024')
container_name = os.environ.get('AZURE_BLOB_CONTAINER', 'raw')
blob_name = f'cms/medicare-inpatient/reporting_year={year}/medicare_inpatient_{year}.json'
service = BlobServiceClient.from_connection_string(os.environ['AZURE_STORAGE_CONNECTION_STRING'])
blob = service.get_blob_client(container=container_name, blob=blob_name)
raw_bytes = blob.download_blob().readall()
metadata = blob.get_blob_properties().metadata
assert hashlib.sha256(raw_bytes).hexdigest() == metadata['sha256']
records = json.loads(raw_bytes)
raw_df = pd.DataFrame(records)
assert raw_df.shape == (int(metadata['row_count']), int(metadata['column_count']))
print(f'Blob: {container_name}/{blob_name}')
print(f'SHA-256: {metadata["sha256"]}')
print(f'Shape: {raw_df.shape}')

In [ ]:
expected_columns = ['Rndrng_Prvdr_CCN', 'Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_St', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_RUCA_Desc', 'DRG_Cd', 'DRG_Desc', 'Tot_Dschrgs', 'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt']
assert set(raw_df.columns) == set(expected_columns)
display(pd.Series(raw_df.columns, name='source_column').to_frame())

## Schema Inspection

A pandas `object` dtype describes the in-memory storage representation. Business meaning comes from the CMS definitions: CCN, FIPS, ZIP, RUCA, and DRG are identifiers or categories even when they contain digits.

In [ ]:
display(raw_df.dtypes.rename('storage_dtype').to_frame())
raw_df.info()

## Missing Values and Cardinality

In [ ]:
missing_mask = raw_df.isna() | raw_df.eq('')
profile = pd.DataFrame({
    'missing_count': missing_mask.sum(),
    'missing_percentage': missing_mask.mean().mul(100),
    'distinct_count': raw_df.replace('', pd.NA).nunique(dropna=True),
})
display(profile)

## Duplicate Validation

In [ ]:
candidate_key = ['Rndrng_Prvdr_CCN', 'DRG_Cd']
duplicate_rows = raw_df.duplicated(candidate_key, keep=False)
key_summary = {
    'rows': len(raw_df),
    'unique_provider_drg_keys': raw_df[candidate_key].drop_duplicates().shape[0],
    'duplicate_rows': int(duplicate_rows.sum()),
    'missing_key_rows': int(raw_df[candidate_key].isna().any(axis=1).sum()),
}
display(pd.Series(key_summary, name='value').to_frame())
assert key_summary['duplicate_rows'] == 0 and key_summary['missing_key_rows'] == 0

## Numeric Profiling

In [ ]:
numeric_columns = ['Tot_Dschrgs', 'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt']
numeric_df = raw_df[numeric_columns].copy()
for column in numeric_columns:
    numeric_df[column] = pd.to_numeric(
        numeric_df[column].astype('string').str.replace(r'[$,]', '', regex=True),
        errors='coerce',
    )
source_numeric_missing = missing_mask[numeric_columns].sum()
parse_failures = numeric_df.isna().sum() - source_numeric_missing
display(parse_failures.rename('new_nulls_after_numeric_parse').to_frame())
display(numeric_df.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

## Consistency Checks

These checks identify confirmed formatting issues or potential anomalies. A large or unusual payment is not an error solely because it is extreme.

In [ ]:
checks = {
    'nonpositive_discharge_rows': int((numeric_df['Tot_Dschrgs'] <= 0).sum()),
    'negative_monetary_rows': int((numeric_df[numeric_columns[1:]] < 0).any(axis=1).sum()),
    'provider_ccn_format_rows': int((~raw_df['Rndrng_Prvdr_CCN'].astype('string').str.fullmatch(r'\d{6}', na=False)).sum()),
    'drg_code_format_rows': int((~raw_df['DRG_Cd'].astype('string').str.fullmatch(r'\d{3}', na=False)).sum()),
    'state_abbreviation_format_rows': int((~raw_df['Rndrng_Prvdr_State_Abrvtn'].astype('string').str.fullmatch(r'[A-Z]{2}', na=False)).sum()),
    'missing_identifier_rows': int(raw_df[candidate_key].isna().any(axis=1).sum()),
    'medicare_payment_above_total_payment_rows': int((numeric_df['Avg_Mdcr_Pymt_Amt'] > numeric_df['Avg_Tot_Pymt_Amt']).sum()),
}
display(pd.Series(checks, name='row_count').to_frame())
display(raw_df[['Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_State_Abrvtn']].drop_duplicates().sort_values(['Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_State_FIPS']))

## Findings

The following statements are generated only from the executed checks. Confirmed issues are distinguished from potential anomalies that warrant investigation.

In [ ]:
print(f"Confirmed: {key_summary['rows']:,} rows, {raw_df.shape[1]} columns, and no duplicate provider-DRG keys.")
print(f"Confirmed: {int(profile['missing_count'].sum()):,} missing source values across all columns.")
print(f"Confirmed issue count: {int(parse_failures.clip(lower=0).sum()):,} values failed numeric parsing.")
print(f"Potential anomaly requiring investigation: {checks['medicare_payment_above_total_payment_rows']:,} rows have Medicare payment above total payment.")
print('Expected source behavior: provider/DRG combinations with 10 or fewer discharges are suppressed by CMS; absent combinations are not zero activity.')